# 🔌 Day 03a: WebSockets in FastAPI — Real-Time Communication

---

## 🎯 What You'll Master Today
- WebSocket vs HTTP (polling, SSE)
- FastAPI WebSocket endpoints
- Connection manager pattern
- Broadcasting to multiple clients
- Room-based messaging
- Authentication in WebSockets

---

## 🎭 The Walkie-Talkie Analogy

**HTTP** is like sending letters — client sends request, server sends response, done. For every update, client must send another letter.

**WebSocket** is like a walkie-talkie — once connected, both sides can talk anytime. No need to re-establish connection.

```
╔═══════════════════════════════════════════════════════════════════╗
║  WebSocket vs HTTP vs SSE                                        ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  HTTP Polling:  Client asks "any updates?" every N seconds      ║
║                 ❌ Wasteful, high latency, many connections       ║
║                                                                   ║
║  Long Polling:  Client asks, server holds until update           ║
║                 ⚠️ Better, but still one-way at a time           ║
║                                                                   ║
║  SSE:           Server pushes events (one-way: server → client) ║
║                 ✅ Good for notifications, live feeds            ║
║                                                                   ║
║  WebSocket:     Full duplex (both ways, any time)                ║
║                 ✅ Chat, games, live collaboration, trading       ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Basic WebSocket Endpoint
# ============================================================
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.testclient import TestClient

app = FastAPI()

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()  # Complete the handshake
    try:
        while True:
            # Wait for message from client
            data = await websocket.receive_text()
            # Echo it back
            await websocket.send_text(f"Echo: {data}")
    except WebSocketDisconnect:
        print("Client disconnected")

# Test with TestClient
client = TestClient(app)
with client.websocket_connect("/ws") as ws:
    ws.send_text("Hello")
    response = ws.receive_text()
    print(f"Received: {response}")
    
    ws.send_text("World")
    response = ws.receive_text()
    print(f"Received: {response}")

In [ ]:
# ============================================================
# Connection Manager — Handle Multiple Clients
# ============================================================
from typing import Dict, Set

class ConnectionManager:
    """Manages active WebSocket connections."""
    
    def __init__(self):
        self.active_connections: list[WebSocket] = []
    
    async def connect(self, websocket: WebSocket):
        await websocket.accept()
        self.active_connections.append(websocket)
    
    def disconnect(self, websocket: WebSocket):
        self.active_connections.remove(websocket)
    
    async def send_personal(self, message: str, websocket: WebSocket):
        await websocket.send_text(message)
    
    async def broadcast(self, message: str):
        """Send message to ALL connected clients."""
        for connection in self.active_connections:
            await connection.send_text(message)

manager = ConnectionManager()

print("ConnectionManager pattern:")
print("  connect()      → Accept + store connection")
print("  disconnect()   → Remove from active list")
print("  broadcast()    → Send to ALL clients")
print("  send_personal() → Send to ONE client")

In [ ]:
# ============================================================
# Chat Room with Connection Manager
# ============================================================

app2 = FastAPI()

class RoomManager:
    def __init__(self):
        self.rooms: Dict[str, Set[WebSocket]] = {}
    
    async def join_room(self, room: str, websocket: WebSocket):
        await websocket.accept()
        if room not in self.rooms:
            self.rooms[room] = set()
        self.rooms[room].add(websocket)
    
    def leave_room(self, room: str, websocket: WebSocket):
        self.rooms[room].discard(websocket)
        if not self.rooms[room]:
            del self.rooms[room]
    
    async def broadcast_to_room(self, room: str, message: str, exclude: WebSocket = None):
        if room in self.rooms:
            for conn in self.rooms[room]:
                if conn != exclude:
                    await conn.send_text(message)

rooms = RoomManager()

@app2.websocket("/ws/{room_name}/{username}")
async def chat_room(websocket: WebSocket, room_name: str, username: str):
    await rooms.join_room(room_name, websocket)
    await rooms.broadcast_to_room(room_name, f"📢 {username} joined the room")
    try:
        while True:
            data = await websocket.receive_text()
            await rooms.broadcast_to_room(
                room_name, f"{username}: {data}", exclude=websocket
            )
    except WebSocketDisconnect:
        rooms.leave_room(room_name, websocket)
        await rooms.broadcast_to_room(room_name, f"📢 {username} left the room")

# Test
client2 = TestClient(app2)
with client2.websocket_connect("/ws/general/Alice") as ws:
    ws.send_text("Hello everyone!")
    print("✅ Chat room WebSocket works")

In [ ]:
# ============================================================
# JSON Messages — Structured Communication
# ============================================================
import json

app3 = FastAPI()

@app3.websocket("/ws/chat")
async def json_chat(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            # Receive JSON
            data = await websocket.receive_json()
            msg_type = data.get("type", "unknown")
            
            if msg_type == "message":
                await websocket.send_json({
                    "type": "message",
                    "from": "server",
                    "content": f"Got: {data.get('content', '')}",
                })
            elif msg_type == "ping":
                await websocket.send_json({"type": "pong"})
            else:
                await websocket.send_json({
                    "type": "error",
                    "content": f"Unknown type: {msg_type}"
                })
    except WebSocketDisconnect:
        pass

client3 = TestClient(app3)
with client3.websocket_connect("/ws/chat") as ws:
    ws.send_json({"type": "message", "content": "Hello!"})
    response = ws.receive_json()
    print(f"JSON response: {response}")
    
    ws.send_json({"type": "ping"})
    response = ws.receive_json()
    print(f"Ping response: {response}")

In [ ]:
# ============================================================
# WebSocket Authentication
# ============================================================

print("""
# WebSocket auth — two approaches:

# APPROACH 1: Query parameter (simple)
@app.websocket("/ws")
async def ws_auth_query(websocket: WebSocket, token: str = Query(...)):
    user = verify_token(token)
    if not user:
        await websocket.close(code=4001)
        return
    await websocket.accept()
    ...

# Client: ws://localhost:8000/ws?token=eyJhbG...


# APPROACH 2: First message auth (more secure)
@app.websocket("/ws")
async def ws_auth_first_message(websocket: WebSocket):
    await websocket.accept()
    # First message must be auth token
    token_data = await asyncio.wait_for(
        websocket.receive_json(), timeout=5.0
    )
    user = verify_token(token_data.get("token"))
    if not user:
        await websocket.send_json({"error": "Invalid token"})
        await websocket.close()
        return
    await websocket.send_json({"status": "authenticated"})
    # Now handle messages...
""")

print("Auth approaches:")
print("  Query param  → Simple but token in URL (logs, history)")
print("  First message → More secure, token not in URL")
print("  Cookie-based  → Automatic if same-origin")

---

## 🗺️ WebSocket Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  WEBSOCKET DECISION GUIDE                                        ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  USE CASE → TECHNOLOGY:                                          ║
║    Notifications (server→client)   → SSE                        ║
║    Chat, games, collaboration      → WebSocket                   ║
║    Occasional updates              → HTTP Polling                ║
║    RESTful data fetching           → Regular HTTP                ║
║                                                                   ║
║  PATTERNS:                                                       ║
║    ConnectionManager  → Track active connections                ║
║    RoomManager        → Group connections by room               ║
║    JSON protocol      → Structured {type, content} messages    ║
║    Heartbeat/ping     → Detect stale connections                ║
║                                                                   ║
║  AUTH:                                                           ║
║    Query param  → ws://host/ws?token=xxx  (simple)              ║
║    First message → Send token as first JSON (secure)            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | WebSocket vs HTTP? | HTTP: request-response. WS: full-duplex, persistent connection |
| 2 | WebSocket vs SSE? | WS: bidirectional. SSE: server→client only, simpler, auto-reconnect |
| 3 | When to use WebSockets? | Real-time: chat, games, live collab, trading, live dashboards |
| 4 | How does WS handshake work? | HTTP upgrade request → server accepts → connection stays open |
| 5 | How to scale WebSockets? | Sticky sessions + Redis pub/sub for cross-server broadcasting |
| 6 | How to handle disconnects? | try/except WebSocketDisconnect, heartbeat pings, reconnect on client |
| 7 | WebSocket authentication? | Query param token or first-message auth. Can't use standard headers |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • WebSocket = full-duplex persistent connection         │
## │  • ConnectionManager tracks active connections           │
## │  • broadcast() sends to all, rooms for grouping         │
## │  • Use JSON protocol for structured messages             │
## │  • Auth via query param or first-message token          │
## │  • Scale with Redis pub/sub across servers              │
## │  • Always handle WebSocketDisconnect                    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Background Tasks** — FastAPI's built-in lightweight processing